In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class UNet(nn.Module):
    def __init__(self):
        super(UNet, self).__init__()

        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.ReLU(),
                nn.Conv2d(out_c, out_c, 3, padding=1),
                nn.ReLU()
            )

        self.enc1 = conv_block(3, 64)
        self.enc2 = conv_block(64, 128)

        self.pool = nn.MaxPool2d(2)

        self.bottleneck = conv_block(128, 256)

        self.up2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = conv_block(256, 128)

        self.up1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = conv_block(128, 64)

        self.out = nn.Conv2d(64, 1, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))

        b = self.bottleneck(self.pool(e2))

        d2 = self.up2(b)
        d2 = torch.cat([d2, e2], dim=1)
        d2 = self.dec2(d2)

        d1 = self.up1(d2)
        d1 = torch.cat([d1, e1], dim=1)
        d1 = self.dec1(d1)

        return torch.sigmoid(self.out(d1))

In [ ]:
documents = {
    "Admissions": {
        "Eligibility": "Minimum 60 percent in 12th grade.",
        "Fees": "Annual tuition fee is 2 lakh INR."
    },
    "Hostel": {
        "Rooms": "Single and shared rooms available.",
        "Rules": "Entry allowed before 10 PM."
    }
}

In [ ]:
def hierarchical_retrieval(query):
    for category in documents:
        if category.lower() in query.lower():
            for section in documents[category]:
                if section.lower() in query.lower():
                    return documents[category][section]
            # If category matches but no section matches, return information not found for that category
            return "Information not found."
    # If no category matches at all
    return "Information not found."

In [ ]:
query = "What is the fees in admissions?"
context = hierarchical_retrieval(query)
response = "Based on university records: " + context
print(response)

Based on university records: Annual tuition fee is 2 lakh INR.


In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

data = {
    "hour": np.random.randint(0, 24, 1000),
    "day": np.random.randint(1, 8, 1000),
    "weather": np.random.choice(["Sunny", "Rainy", "Cloudy"], 1000),
    "vehicle_count": np.random.randint(50, 500, 1000)
}

df = pd.DataFrame(data)
df.to_csv("traffic_data.csv", index=False)

print(df.head())

   hour  day weather  vehicle_count
0     6    7   Rainy            131
1    19    1  Cloudy            452
2    14    2   Sunny            208
3    10    6   Sunny            237
4     7    6   Sunny            109


In [ ]:
def vae_loss(recon_x, x, mu, logvar):
    BCE = F.binary_cross_entropy(recon_x, x, reduction='sum')
    KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return BCE + KLD

In [ ]:
import torch

x = torch.randn(1, 3, 64, 64)

for t in range(10):
    noise = torch.randn_like(x)
    x = x - 0.1 * noise

    print("Output shape:", x.shape)

Output shape: torch.Size([1, 3, 64, 64])
Output shape: torch.Size([1, 3, 64, 64])
Output shape: torch.Size([1, 3, 64, 64])
Output shape: torch.Size([1, 3, 64, 64])
Output shape: torch.Size([1, 3, 64, 64])
Output shape: torch.Size([1, 3, 64, 64])
Output shape: torch.Size([1, 3, 64, 64])
Output shape: torch.Size([1, 3, 64, 64])
Output shape: torch.Size([1, 3, 64, 64])
Output shape: torch.Size([1, 3, 64, 64])
